# cf6.ipynb

목표:

1. `cf2/*.json`의 YOLO 객체 검출 결과와 원본 mp4를 결합한다.
2. SegFormer Cityscapes segmentation으로 `road / sidewalk`를 판정하되, cf5보다 인도주행 recall을 높이기 위해 `vehicle bbox`와 `person+vehicle union bbox`를 함께 사용한다.
3. `ms_`, `bs_`, `es_` 계열 인도주행 GT 그룹과 `md_` 헬멧미착용 GT 그룹을 분리해 평가한다.
4. `cf5_output/frame_anomaly_results.json`과 `cf6_output/frame_anomaly_results.json`을 비교해 감지가 증가했는지/감소했는지 확인한다.
5. Roboflow/로컬 YOLO 헬멧 모델 연결 구조는 유지한다.


In [1]:
# Cell 1. 기본 설정
from pathlib import Path
import os
import json
import math
import time
import gc
import tempfile
from collections import Counter, defaultdict

ROOT = Path.cwd()
CF2_DIR = ROOT / "cf2"
OUT_DIR = ROOT / "cf6_output"
DEBUG_DIR = ROOT / "cf6_debug"
HELMET_CROP_DIR = DEBUG_DIR / "helmet_crops"
OUT_DIR.mkdir(exist_ok=True)
DEBUG_DIR.mkdir(exist_ok=True)
HELMET_CROP_DIR.mkdir(parents=True, exist_ok=True)

# 네가 알려준 원본 영상 루트
VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")

# RTX 3060이 CUDA로 잡히면 b5 사용. CPU면 b0로 자동 fallback.
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"

# ===== 헬멧 모델 설정 =====
# 1순위 후보: Roboflow Bike Helmet Detection 계열
# Universe URL 예시: https://universe.roboflow.com/bike-helmets/bike-helmet-detection-2vdjo
# Roboflow hosted inference를 쓰려면 환경변수 ROBOFLOW_API_KEY를 설정한다.
USE_ROBOFLOW_HOSTED_INFERENCE = False
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
ROBOFLOW_MODEL_ID = "bike-helmet-detection-2vdjo/1"  # 버전이 다르면 Roboflow API Docs의 model_id로 수정

# 로컬 YOLO .pt가 있다면 여기에 연결한다.
# 예: HELMET_MODEL_PATH = r"C:\Users\우성한\models\bike_helmet_best.pt"
HELMET_MODEL_PATH = None

# Roboflow dataset export 후 직접 학습할 경우 저장 위치
RF_WORKSPACE = "bike-helmets"
RF_PROJECT = "bike-helmet-detection-2vdjo"
RF_VERSION = 1
RF_EXPORT_FORMAT = "yolov8"
RF_DATASET_DIR = ROOT / "roboflow_datasets"
RF_TRAIN_PROJECT_DIR = ROOT / "runs_cf6_helmet"

# 전체 처리 제한. 처음 테스트할 때는 작게 두고, 검증 후 None으로 변경.
MAX_JSON_FILES = None
MAX_FRAMES_PER_VIDEO = None
# MAX_JSON_FILES = 2
# MAX_FRAMES_PER_VIDEO = 120

# 사람/이륜차 매칭 설정
PERSON_CLASSES = {"person", "rider"}
VEHICLE_CLASSES = {"bicycle", "motorcycle"}
MIN_PERSON_CONF = 0.10
MIN_VEHICLE_CONF = 0.10
MAX_CENTER_DISTANCE = 280
MIN_OVERLAP_RATIO = 0.015

# Cityscapes class id
ROAD_ID = 0
SIDEWALK_ID = 1

# 인도/차도 판정 보강 설정
MASK_PATCH_RADIUS = 16
DILATE_KERNEL_SIZE = 9      # sidewalk/road 경계 보완. 0 또는 1이면 dilation 거의 없음.
BOTTOM_BAND_Y1 = 0.72       # vehicle bbox 하단부 band 시작 비율
BOTTOM_BAND_Y2 = 1.00
BOTTOM_BAND_X_MARGIN = 0.08
SIDEWALK_SCORE_THRESHOLD = 0.18
ROAD_SCORE_THRESHOLD = 0.18
SIDEWALK_MARGIN_OVER_ROAD = 0.03
ROAD_MARGIN_OVER_SIDEWALK = 0.02

# 헬멧 판정 설정
HELMET_CONF = 0.15
SAVE_HELMET_CROPS = True
MAX_SAVED_HELMET_CROPS = 300

# 성능 설정
USE_AMP_ON_CUDA = True
CLEAR_CUDA_CACHE_EVERY_N_VIDEOS = 3
SAVE_PARTIAL_EVERY_N_VIDEOS = 5

# cf5 대비 성능 변화 확인용 baseline 경로
BASELINE_CF5_RESULT_PATH = ROOT / "cf5_output" / "frame_anomaly_results.json"

# 평가용 prefix. 최종 판정에 정답 라벨을 직접 쓰지는 않고, 평가/비교에만 사용한다.
SIDEWALK_GT_PREFIXES = ("ms_", "bs_", "es_")
HELMET_GT_PREFIXES = ("md_",)

# cf6 인도 판정 알고리즘 버전
SIDEWALK_JUDGE_VERSION = "v2_vehicle_union_bbox"

print("ROOT:", ROOT)
print("CF2_DIR exists:", CF2_DIR.exists(), CF2_DIR)
print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("OUT_DIR:", OUT_DIR)
print("BASELINE_CF5_RESULT_PATH exists:", BASELINE_CF5_RESULT_PATH.exists(), BASELINE_CF5_RESULT_PATH)
print("ROBOFLOW_API_KEY set:", bool(ROBOFLOW_API_KEY))


ROOT: C:\Users\우성한
CF2_DIR exists: True C:\Users\우성한\cf2
VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
OUT_DIR: C:\Users\우성한\cf6_output
BASELINE_CF5_RESULT_PATH exists: False C:\Users\우성한\cf5_output\frame_anomaly_results.json
ROBOFLOW_API_KEY set: True


In [2]:
# Cell 2. 의존성 및 CUDA 확인
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    import torch
    print("torch OK:", torch.__version__)
    print("torch cuda version:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("gpu memory total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
        torch.backends.cudnn.benchmark = True
except Exception as e:
    missing.append("torch")
    print("torch import failed:", repr(e))

try:
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    print("transformers OK")
except Exception as e:
    missing.append("transformers")
    print("transformers import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

try:
    from PIL import Image
    print("PIL OK")
except Exception as e:
    missing.append("pillow")
    print("PIL import failed:", repr(e))

if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy torch transformers ultralytics pillow roboflow inference-sdk")


cv2 OK
numpy OK
torch OK: 2.6.0+cu126
torch cuda version: 12.6
cuda available: True
gpu: NVIDIA GeForce RTX 3060
gpu memory total GB: 12.0
transformers OK
ultralytics OK
PIL OK


## Roboflow 모델/데이터셋 연결 방식

아래 셀은 실제 다운로드/연결용이다.

- API 키가 있으면 `ROBOFLOW_API_KEY` 환경변수를 설정한 뒤 실행한다.
- hosted inference만 쓰려면 `USE_ROBOFLOW_HOSTED_INFERENCE=True`로 바꾼다.
- `.pt` 파일을 직접 쓰려면 `HELMET_MODEL_PATH`에 실제 `best.pt` 경로를 넣는다.
- Roboflow에서 dataset export만 되는 경우에는 Cell 3B로 내려받고 Cell 3C로 짧게 학습해 `best.pt`를 만든다.


In [3]:
# Cell 3A. Roboflow hosted inference client 준비: 선택 실행
rf_client = None
rf_hosted_ready = False

if USE_ROBOFLOW_HOSTED_INFERENCE:
    if not ROBOFLOW_API_KEY:
        print("ROBOFLOW_API_KEY가 없어 hosted inference를 사용할 수 없습니다.")
    else:
        try:
            from inference_sdk import InferenceHTTPClient
            rf_client = InferenceHTTPClient(
                api_url="https://detect.roboflow.com",
                api_key=ROBOFLOW_API_KEY,
            )
            rf_hosted_ready = True
            print("Roboflow hosted inference ready:", ROBOFLOW_MODEL_ID)
        except Exception as e:
            print("Roboflow hosted inference init failed:", repr(e))
else:
    print("USE_ROBOFLOW_HOSTED_INFERENCE=False -> 로컬 .pt 또는 no-model fallback 사용")


USE_ROBOFLOW_HOSTED_INFERENCE=False -> 로컬 .pt 또는 no-model fallback 사용


In [4]:
# Cell 3B. Roboflow 데이터셋 다운로드: API 키가 있을 때만 선택 실행
# 실행 전 RF_VERSION이 실제 Roboflow 프로젝트의 버전과 맞는지 확인한다.
# 성공하면 data.yaml 경로가 출력된다.

def download_roboflow_dataset():
    if not ROBOFLOW_API_KEY:
        raise RuntimeError("ROBOFLOW_API_KEY가 없습니다. 환경변수 설정 후 다시 실행하세요.")
    try:
        from roboflow import Roboflow
    except Exception as e:
        raise RuntimeError("roboflow 패키지가 없습니다. pip install roboflow 후 실행하세요.") from e

    RF_DATASET_DIR.mkdir(exist_ok=True)
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    version = project.version(RF_VERSION)
    dataset = version.download(RF_EXPORT_FORMAT, location=str(RF_DATASET_DIR / f"{RF_PROJECT}-{RF_VERSION}"))
    data_yaml = Path(dataset.location) / "data.yaml"
    print("dataset location:", dataset.location)
    print("data.yaml exists:", data_yaml.exists(), data_yaml)
    return data_yaml

# 필요할 때 아래 주석 해제
# RF_DATA_YAML = download_roboflow_dataset()


In [5]:
# Cell 3C. Roboflow export 데이터셋으로 YOLOv8 직접 학습: 선택 실행
# Cell 3B로 RF_DATA_YAML을 얻은 뒤 실행한다.
# RTX 3060 기준으로 처음에는 epochs를 작게 두고 정상 동작만 확인한다.

def train_helmet_yolov8(data_yaml, epochs=30, imgsz=640, batch=8, model="yolov8n.pt"):
    if YOLO is None:
        raise RuntimeError("ultralytics가 없습니다. pip install ultralytics 후 실행하세요.")
    data_yaml = Path(data_yaml)
    if not data_yaml.exists():
        raise FileNotFoundError(data_yaml)

    model_obj = YOLO(model)
    result = model_obj.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project=str(RF_TRAIN_PROJECT_DIR),
        name="bike_helmet_yolov8",
        device=0 if torch.cuda.is_available() else "cpu",
    )
    best_pt = RF_TRAIN_PROJECT_DIR / "bike_helmet_yolov8" / "weights" / "best.pt"
    print("training done. best.pt:", best_pt, "exists:", best_pt.exists())
    return best_pt

# 필요할 때 아래 주석 해제
# HELMET_MODEL_PATH = str(train_helmet_yolov8(RF_DATA_YAML, epochs=30, imgsz=640, batch=8))


In [6]:
# Cell 4. JSON/영상 파일 인덱싱
if not CF2_DIR.exists():
    raise FileNotFoundError(f"cf2 폴더가 없습니다: {CF2_DIR}")
if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트가 없습니다: {VIDEO_ROOT}")

json_files = sorted(CF2_DIR.glob("*.json"))
if MAX_JSON_FILES is not None:
    json_files = json_files[:MAX_JSON_FILES]
print("json files:", len(json_files))
print([p.name for p in json_files[:10]])

print("\n원본 mp4 인덱싱 중...")
t0 = time.time()
video_index = {}
for p in VIDEO_ROOT.rglob("*.mp4"):
    if p.name not in video_index:
        video_index[p.name] = p
print("indexed videos:", len(video_index), "elapsed_sec:", round(time.time() - t0, 2))

def find_video_path(video_name: str) -> Path:
    p = video_index.get(video_name)
    if p is not None:
        return p
    stem = Path(video_name).stem
    candidates = [v for k, v in video_index.items() if Path(k).stem == stem]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"video not found for {video_name}")


json files: 10
['bs_e4387_c1.json', 'bs_e4387_c2.json', 'es_e3768_c1.json', 'es_e3768_c2.json', 'md_e2122_c1.json', 'md_e2122_c2.json', 'mp_e2730_c1.json', 'mp_e2730_c2.json', 'ms_e3500_c1.json', 'ms_e3500_c2.json']

원본 mp4 인덱싱 중...
indexed videos: 22 elapsed_sec: 0.01


In [7]:
# Cell 5. cf2 JSON 구조 확인
if not json_files:
    raise RuntimeError("cf2 JSON 파일이 없습니다.")

sample_path = json_files[0]
with open(sample_path, encoding="utf-8") as f:
    sample = json.load(f)

print("sample file:", sample_path.name)
print("top keys:", sample.keys())
print("video_name:", sample.get("video_name"))
print("summary:", sample.get("summary"))
print("frame count:", len(sample.get("frames", [])))

non_empty = [fr for fr in sample.get("frames", []) if fr.get("object_count", 0) > 0]
print("non-empty frame count:", len(non_empty))
if non_empty:
    fr = non_empty[0]
    print("first non-empty frame_id:", fr.get("frame_id"))
    print("classes type:", type(fr.get("classes")))
    print("classes:", fr.get("classes"))
    print("first object:", fr.get("objects", [None])[0])


sample file: bs_e4387_c1.json
top keys: dict_keys(['video_name', 'summary', 'frames'])
video_name: bs_e4387_c1.mp4
summary: {'bicycle': 22, 'person': 79, 'motorcycle': 2}
frame count: 181
non-empty frame count: 68
first non-empty frame_id: 55
classes type: <class 'list'>
classes: ['bicycle']
first object: {'class': 'bicycle', 'confidence': 0.662, 'bbox': [1209, 931, 1303, 1078]}


In [8]:
# Cell 6. bbox 및 rider-vehicle 매칭 함수

def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bottom_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, y2)


def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x1 = max(ax1, bx1)
    y1 = max(ay1, by1)
    x2 = min(ax2, bx2)
    y2 = min(ay2, by2)
    return max(0, x2 - x1) * max(0, y2 - y1)


def dist(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def extract_objects(frame):
    persons = []
    vehicles = []
    for obj in frame.get("objects", []):
        cls = obj.get("class")
        conf = float(obj.get("confidence", 0.0))
        bbox = obj.get("bbox")
        if not bbox or len(bbox) != 4:
            continue
        clean = {
            "class": cls,
            "confidence": round(conf, 4),
            "bbox": [int(round(v)) for v in bbox],
        }
        if cls in PERSON_CLASSES and conf >= MIN_PERSON_CONF:
            persons.append(clean)
        elif cls in VEHICLE_CLASSES and conf >= MIN_VEHICLE_CONF:
            vehicles.append(clean)
    return persons, vehicles


def match_riders(persons, vehicles):
    pairs = []
    used_persons = set()
    for vehicle_idx, vehicle in enumerate(vehicles):
        vb = vehicle["bbox"]
        vcenter = bbox_center(vb)
        best = None
        for person_idx, person in enumerate(persons):
            if person_idx in used_persons:
                continue
            pb = person["bbox"]
            pc = bbox_center(pb)
            d = dist(pc, vcenter)
            inter = intersection_area(pb, vb)
            overlap = inter / max(1, min(bbox_area(pb), bbox_area(vb)))
            vertical_ok = pc[1] <= vb[3] + 100
            close_ok = d <= MAX_CENTER_DISTANCE
            overlap_ok = overlap >= MIN_OVERLAP_RATIO
            if not vertical_ok or not (close_ok or overlap_ok):
                continue
            score = overlap * 2.0 + max(0, 1 - d / max(1, MAX_CENTER_DISTANCE))
            if best is None or score > best["match_score"]:
                best = {
                    "person_index": person_idx,
                    "vehicle_index": vehicle_idx,
                    "person": person,
                    "vehicle": vehicle,
                    "distance": round(d, 2),
                    "overlap": round(overlap, 4),
                    "match_score": round(score, 4),
                }
        if best is not None:
            used_persons.add(best["person_index"])
            pairs.append(best)
    return pairs


In [9]:
# Cell 7. SegFormer 로드: CUDA면 b5, CPU면 b0 fallback
import torch
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEG_MODEL_ID = SEG_MODEL_ID_GPU if DEVICE.type == "cuda" else SEG_MODEL_ID_CPU
print("DEVICE:", DEVICE)
print("SEG_MODEL_ID:", SEG_MODEL_ID)

if DEVICE.type != "cuda":
    print("경고: CUDA가 잡히지 않아 CPU로 실행합니다. b5 전체 처리는 매우 느리므로 CPU에서는 b0와 제한 처리 권장.")

seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
seg_model.to(DEVICE)
seg_model.eval()

print("SegFormer loaded")
print("model device:", next(seg_model.parameters()).device)
print("id2label:", seg_model.config.id2label)


DEVICE: cuda
SEG_MODEL_ID: nvidia/segformer-b5-finetuned-cityscapes-1024-1024


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

SegFormer loaded
model device: cuda:0
id2label: {0: 'road', 1: 'sidewalk', 2: 'building', 3: 'wall', 4: 'fence', 5: 'pole', 6: 'traffic light', 7: 'traffic sign', 8: 'vegetation', 9: 'terrain', 10: 'sky', 11: 'person', 12: 'rider', 13: 'car', 14: 'truck', 15: 'bus', 16: 'train', 17: 'motorcycle', 18: 'bicycle'}


In [10]:
# Cell 8. 헬멧 YOLO 로컬 모델 로드
helmet_model = None
helmet_model_loaded = False
helmet_backend = "none"

if HELMET_MODEL_PATH is None:
    print("HELMET_MODEL_PATH is None -> 로컬 YOLO 헬멧 모델은 로드하지 않음")
elif YOLO is None:
    print("ultralytics가 없어 로컬 YOLO 헬멧 모델을 로드할 수 없음")
else:
    helmet_path = Path(HELMET_MODEL_PATH)
    if not helmet_path.exists():
        print("helmet model file not found:", helmet_path)
    else:
        helmet_model = YOLO(str(helmet_path))
        try:
            helmet_model.to(str(DEVICE))
        except Exception as e:
            print("helmet_model.to failed, continue:", repr(e))
        helmet_model_loaded = True
        helmet_backend = "local_yolo"
        print("helmet model loaded:", helmet_path)
        try:
            print("helmet model names:", helmet_model.names)
        except Exception:
            pass

if rf_hosted_ready:
    helmet_backend = "roboflow_hosted"
    print("helmet backend switched to Roboflow hosted inference:", ROBOFLOW_MODEL_ID)

print("helmet_backend:", helmet_backend)


HELMET_MODEL_PATH is None -> 로컬 YOLO 헬멧 모델은 로드하지 않음
helmet_backend: none


In [11]:
# Cell 9. 프레임/SegFormer/인도 판정 함수 - cf6 v2
import cv2
import numpy as np
from PIL import Image


def read_frame_from_cap(cap, frame_id: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ret, frame_bgr = cap.read()
    if not ret:
        return None
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def segment_frame(image_rgb: np.ndarray):
    """원본 해상도 pred map 반환. pred 값은 Cityscapes class id."""
    pil_image = Image.fromarray(image_rgb)
    inputs = seg_processor(images=pil_image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        if DEVICE.type == "cuda" and USE_AMP_ON_CUDA:
            with torch.cuda.amp.autocast():
                outputs = seg_model(**inputs)
        else:
            outputs = seg_model(**inputs)

    logits = outputs.logits
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image_rgb.shape[:2],
        mode="bilinear",
        align_corners=False,
    )
    pred = upsampled_logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    return pred


def make_class_masks(pred):
    road = (pred == ROAD_ID).astype(np.uint8)
    sidewalk = (pred == SIDEWALK_ID).astype(np.uint8)
    if DILATE_KERNEL_SIZE and DILATE_KERNEL_SIZE > 1:
        k = np.ones((DILATE_KERNEL_SIZE, DILATE_KERNEL_SIZE), np.uint8)
        road_d = cv2.dilate(road, k, iterations=1).astype(bool)
        sidewalk_d = cv2.dilate(sidewalk, k, iterations=1).astype(bool)
    else:
        road_d = road.astype(bool)
        sidewalk_d = sidewalk.astype(bool)
    return road.astype(bool), sidewalk.astype(bool), road_d, sidewalk_d


def safe_crop_mask(mask, x1, y1, x2, y2):
    h, w = mask.shape[:2]
    x1 = max(0, min(w, int(round(x1))))
    x2 = max(0, min(w, int(round(x2))))
    y1 = max(0, min(h, int(round(y1))))
    y2 = max(0, min(h, int(round(y2))))
    if x1 >= x2 or y1 >= y2:
        return None
    return mask[y1:y2, x1:x2]


def mask_patch_ratio(mask, center_xy, radius=MASK_PATCH_RADIUS):
    h, w = mask.shape[:2]
    x, y = center_xy
    x = int(round(x)); y = int(round(y))
    x1 = max(0, x - radius); x2 = min(w, x + radius + 1)
    y1 = max(0, y - radius); y2 = min(h, y + radius + 1)
    if x1 >= x2 or y1 >= y2:
        return 0.0
    return float(mask[y1:y2, x1:x2].mean())


def clip_box(box, w, h):
    x1, y1, x2, y2 = [float(v) for v in box]
    return [
        max(0, min(w - 1, x1)),
        max(0, min(h - 1, y1)),
        max(0, min(w - 1, x2)),
        max(0, min(h - 1, y2)),
    ]


def expand_box(box, w, h, scale_x=0.25, scale_y=0.10):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    return clip_box([
        x1 - bw * scale_x,
        y1 - bh * scale_y,
        x2 + bw * scale_x,
        y2 + bh * scale_y,
    ], w, h)


def union_box(box1, box2, w, h):
    x1 = min(float(box1[0]), float(box2[0]))
    y1 = min(float(box1[1]), float(box2[1]))
    x2 = max(float(box1[2]), float(box2[2]))
    y2 = max(float(box1[3]), float(box2[3]))
    return clip_box([x1, y1, x2, y2], w, h)


def bottom_band_from_box(box, x_margin=0.08, y1_ratio=0.70, y2_ratio=1.00):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    mx = bw * x_margin
    return x1 - mx, y1 + bh * y1_ratio, x2 + mx, y1 + bh * y2_ratio


def support_points_from_box(box):
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    # cf5보다 좌우와 하단 근처 지점을 조금 더 촘촘히 본다.
    xs = [x1 + bw * r for r in [0.10, 0.22, 0.35, 0.50, 0.65, 0.78, 0.90]]
    ys = [y1 + bh * r for r in [0.72, 0.82, 0.92, 0.99]]
    return [(x, y) for y in ys for x in xs]


def score_surface_from_box(road_d, sidewalk_d, box, *, label="box"):
    """
    bbox 하단 band + support points 기반으로 road/sidewalk 점수 계산.
    cf6에서는 vehicle bbox뿐 아니라 person+vehicle union bbox에도 적용한다.
    """
    h, w = road_d.shape[:2]
    x1, y1, x2, y2 = clip_box(box, w, h)
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0, f"{label}: invalid_box"

    bx1, by1, bx2, by2 = bottom_band_from_box(
        [x1, y1, x2, y2],
        x_margin=BOTTOM_BAND_X_MARGIN,
        y1_ratio=0.70,
        y2_ratio=1.00,
    )

    road_band = safe_crop_mask(road_d, bx1, by1, bx2, by2)
    sidewalk_band = safe_crop_mask(sidewalk_d, bx1, by1, bx2, by2)
    road_band_ratio = float(road_band.mean()) if road_band is not None else 0.0
    sidewalk_band_ratio = float(sidewalk_band.mean()) if sidewalk_band is not None else 0.0

    pts = support_points_from_box([x1, y1, x2, y2])
    road_point_ratios = [mask_patch_ratio(road_d, p) for p in pts]
    sidewalk_point_ratios = [mask_patch_ratio(sidewalk_d, p) for p in pts]
    road_point_score = float(np.mean(road_point_ratios)) if road_point_ratios else 0.0
    sidewalk_point_score = float(np.mean(sidewalk_point_ratios)) if sidewalk_point_ratios else 0.0

    road_score = 0.60 * road_band_ratio + 0.40 * road_point_score
    sidewalk_score = 0.60 * sidewalk_band_ratio + 0.40 * sidewalk_point_score

    reason = (
        f"{label}: band road={road_band_ratio:.3f}, sidewalk={sidewalk_band_ratio:.3f}; "
        f"points road={road_point_score:.3f}, sidewalk={sidewalk_point_score:.3f}; "
        f"score road={road_score:.3f}, sidewalk={sidewalk_score:.3f}"
    )
    return road_score, sidewalk_score, reason


def judge_sidewalk_v2(pred, person_bbox, vehicle_bbox):
    """
    cf6 v2 인도 주행 판정.

    개선점:
    1. vehicle bbox 하단만 보지 않고 person+vehicle union bbox 하단도 함께 본다.
    2. bbox를 약간 확장해서 바퀴/접지점이 bbox 밖에 걸리는 경우를 보완한다.
    3. strong sidewalk, weak sidewalk candidate, road, unknown을 분리한다.
    4. prefix 정답 라벨은 사용하지 않는다. prefix는 뒤 평가 셀에서만 사용한다.
    """
    road, sidewalk, road_d, sidewalk_d = make_class_masks(pred)
    h, w = pred.shape[:2]

    vehicle_expanded = expand_box(vehicle_bbox, w, h, scale_x=0.30, scale_y=0.10)
    pv_union = union_box(person_bbox, vehicle_bbox, w, h)
    union_expanded = expand_box(pv_union, w, h, scale_x=0.15, scale_y=0.05)

    vr, vs, vehicle_reason = score_surface_from_box(road_d, sidewalk_d, vehicle_expanded, label="vehicle_expanded")
    ur, us, union_reason = score_surface_from_box(road_d, sidewalk_d, union_expanded, label="person_vehicle_union")

    # 최종 점수: vehicle evidence를 기본으로, union evidence가 인도 근거를 보완하도록 사용한다.
    # road는 union 하단이 사람 다리/몸통 쪽으로 섞일 수 있어 0.85 가중만 준다.
    road_score = max(vr, ur * 0.85)
    sidewalk_score = max(vs, us)

    reason = (
        f"version={SIDEWALK_JUDGE_VERSION}; "
        f"final road={road_score:.3f}, sidewalk={sidewalk_score:.3f}; "
        f"{vehicle_reason}; {union_reason}"
    )

    # cf5보다 recall을 올리되, md/other에서 오탐 폭증을 막기 위해 road 우세 조건을 남겨둔다.
    if sidewalk_score >= 0.18 and sidewalk_score >= road_score * 0.65:
        return "sidewalk_riding", "strong_sidewalk_surface_v2; " + reason, road_score, sidewalk_score

    if sidewalk_score >= 0.10 and sidewalk_score >= road_score * 0.45:
        return "sidewalk_candidate", "weak_sidewalk_surface_v2; " + reason, road_score, sidewalk_score

    if sidewalk_score > 0.05 and sidewalk_score > road_score:
        return "sidewalk_candidate", "sidewalk_more_than_road_v2; " + reason, road_score, sidewalk_score

    if road_score >= 0.20 and road_score > sidewalk_score * 1.40:
        return "road_riding", "road_surface_v2; " + reason, road_score, sidewalk_score

    return "unknown", "low_surface_confidence_v2; " + reason, road_score, sidewalk_score


# 기존 코드 호환용 wrapper. cf6 메인 루프에서는 judge_sidewalk_v2를 직접 호출한다.
def judge_sidewalk(pred, vehicle_bbox):
    return judge_sidewalk_v2(pred, vehicle_bbox, vehicle_bbox)


In [12]:
# Cell 10. 헬멧 판정 함수: local YOLO + Roboflow hosted inference 지원
saved_crop_count = 0


def crop_head_and_upper_body(image_rgb: np.ndarray, person_bbox, mode="head"):
    if image_rgb is None:
        return None
    h_img, w_img = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in person_bbox]
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 0 or bh <= 0:
        return None

    if mode == "head":
        # CCTV에서 머리 bbox가 작게 잡히는 문제를 줄이기 위해 머리 주변을 넉넉히 crop
        cy2 = y1 + int(bh * 0.45)
        mx = int(bw * 0.35)
        my = int(bh * 0.12)
    else:
        # head crop 실패 시 상반신 crop으로 재시도
        cy2 = y1 + int(bh * 0.62)
        mx = int(bw * 0.25)
        my = int(bh * 0.08)

    cx1 = max(0, x1 - mx)
    cy1 = max(0, y1 - my)
    cx2 = min(w_img, x2 + mx)
    cy2 = min(h_img, cy2 + my)
    if cx1 >= cx2 or cy1 >= cy2:
        return None
    return image_rgb[cy1:cy2, cx1:cx2]


def normalize_helmet_label(label: str):
    label = str(label).lower().strip().replace("-", "_").replace(" ", "_")
    no_helmet_labels = {
        "no_helmet", "nohelmet", "without_helmet", "withouthelmet", "bare_head", "barehead",
        "rider_without_helmet", "without", "no-helmet", "not_helmet",
    }
    helmet_labels = {
        "helmet", "with_helmet", "withhelmet", "hardhat", "hard_hat", "rider_with_helmet", "with",
    }
    if label in no_helmet_labels or "without" in label or "no_helmet" in label or "nohelmet" in label:
        return "no_helmet"
    if label in helmet_labels or "with_helmet" in label or "withhelmet" in label:
        return "helmet"
    return label


def save_debug_crop(crop, video_name, frame_id, pair_idx, mode):
    global saved_crop_count
    if not SAVE_HELMET_CROPS or crop is None or saved_crop_count >= MAX_SAVED_HELMET_CROPS:
        return None
    stem = Path(video_name).stem
    path = HELMET_CROP_DIR / f"{stem}_f{frame_id}_p{pair_idx}_{mode}.jpg"
    cv2.imwrite(str(path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    saved_crop_count += 1
    return str(path)


def predict_helmet_local_yolo(crop):
    try:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF, device=0 if DEVICE.type == "cuda" else "cpu")
    except TypeError:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF)

    detections = []
    for res in pred:
        names = getattr(res, "names", getattr(helmet_model, "names", {}))
        boxes = getattr(res, "boxes", None)
        if boxes is None:
            continue
        for b in boxes:
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
            norm = normalize_helmet_label(label)
            xyxy = [int(round(v)) for v in b.xyxy[0].detach().cpu().tolist()]
            detections.append({"label": label, "normalized": norm, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def predict_helmet_roboflow(crop):
    # inference_sdk는 파일 경로 inference가 가장 안정적이라 임시 jpg로 저장한다.
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cv2.imwrite(tmp_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        response = rf_client.infer(tmp_path, model_id=ROBOFLOW_MODEL_ID)
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass

    detections = []
    for p in response.get("predictions", []):
        label = p.get("class", "")
        conf = float(p.get("confidence", 0.0))
        norm = normalize_helmet_label(label)
        x = float(p.get("x", 0)); y = float(p.get("y", 0)); w = float(p.get("width", 0)); h = float(p.get("height", 0))
        xyxy = [int(x - w/2), int(y - h/2), int(x + w/2), int(y + h/2)]
        detections.append({"label": label, "normalized": norm, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def decide_helmet_from_detections(detections):
    if not detections:
        return "unknown", "helmet model returned no detection", []
    recognized = [d for d in detections if d.get("normalized") in {"helmet", "no_helmet"}]
    if not recognized:
        best = max(detections, key=lambda x: x["confidence"])
        return "unknown", f"helmet model labels not recognized: {best['label']}", detections

    # no_helmet은 위험 이벤트라서 비슷한 confidence이면 우선 보존한다.
    no_hits = [d for d in recognized if d["normalized"] == "no_helmet"]
    hel_hits = [d for d in recognized if d["normalized"] == "helmet"]
    best_no = max(no_hits, key=lambda x: x["confidence"], default=None)
    best_hel = max(hel_hits, key=lambda x: x["confidence"], default=None)

    if best_no and (best_hel is None or best_no["confidence"] >= best_hel["confidence"] - 0.05):
        return "no_helmet", f"detected no_helmet conf={best_no['confidence']}", detections
    if best_hel:
        return "helmet", f"detected helmet conf={best_hel['confidence']}", detections
    return "unknown", "no recognized helmet/no_helmet detection", detections


def judge_helmet(image_rgb, person_bbox, video_name=None, frame_id=None, pair_idx=0):
    if helmet_backend == "none":
        return "unknown", "helmet model is not connected. Set HELMET_MODEL_PATH or Roboflow API.", []

    all_detections = []
    crop_paths = []
    for mode in ["head", "upper"]:
        crop = crop_head_and_upper_body(image_rgb, person_bbox, mode=mode)
        if crop is None:
            continue
        crop_path = save_debug_crop(crop, video_name or "video", frame_id or -1, pair_idx, mode)
        if crop_path:
            crop_paths.append(crop_path)

        try:
            if helmet_backend == "local_yolo":
                dets = predict_helmet_local_yolo(crop)
            elif helmet_backend == "roboflow_hosted":
                dets = predict_helmet_roboflow(crop)
            else:
                dets = []
        except Exception as e:
            return "unknown", f"helmet predict failed on {mode}: {repr(e)}", []

        for d in dets:
            d["crop_mode"] = mode
        all_detections.extend(dets)

        status, reason, _ = decide_helmet_from_detections(dets)
        if status in {"helmet", "no_helmet"}:
            if crop_paths:
                reason += f"; crop_saved={crop_paths[-1]}"
            return status, reason, all_detections

    status, reason, detections = decide_helmet_from_detections(all_detections)
    if crop_paths:
        reason += f"; crops_saved={len(crop_paths)}"
    return status, reason, detections


In [13]:
# Cell 11. 전체 처리 실행
results = []
failed_videos = []
start_time_all = time.time()

for json_idx, json_path in enumerate(json_files, start=1):
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    video_name = data.get("video_name")
    if not video_name:
        print("skip: no video_name", json_path.name)
        continue

    try:
        video_path = find_video_path(video_name)
    except Exception as e:
        failed_videos.append({"video_name": video_name, "reason": repr(e)})
        print("video not found:", video_name, repr(e))
        continue

    frames = data.get("frames", [])
    if MAX_FRAMES_PER_VIDEO is not None:
        frames = frames[:MAX_FRAMES_PER_VIDEO]

    print(f"\n[{json_idx}/{len(json_files)}] start {video_name}, frames={len(frames)}, device={DEVICE}, helmet_backend={helmet_backend}")
    t0 = time.time()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        failed_videos.append({"video_name": video_name, "reason": "cv2.VideoCapture failed"})
        print("video open failed:", video_path)
        continue

    video_pair_count = 0
    processed_pair_frames = 0

    for frame_idx, frame in enumerate(frames, start=1):
        frame_id = int(frame.get("frame_id", -1))
        persons, vehicles = extract_objects(frame)
        if not persons or not vehicles:
            continue

        pairs = match_riders(persons, vehicles)
        if not pairs:
            continue

        image_rgb = read_frame_from_cap(cap, frame_id)
        if image_rgb is None:
            print("frame read failed:", video_name, frame_id)
            continue

        try:
            pred = segment_frame(image_rgb)
        except RuntimeError as e:
            if DEVICE.type == "cuda" and "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print("CUDA OOM at", video_name, frame_id, "-> skip frame")
                continue
            print("segmentation failed:", video_name, frame_id, repr(e))
            continue
        except Exception as e:
            print("segmentation failed:", video_name, frame_id, repr(e))
            continue

        processed_pair_frames += 1
        video_pair_count += len(pairs)

        for pair_idx, pair in enumerate(pairs):
            person = pair["person"]
            vehicle = pair["vehicle"]

            sidewalk_status, sidewalk_reason, road_score, sidewalk_score = judge_sidewalk_v2(
                pred,
                person_bbox=person["bbox"],
                vehicle_bbox=vehicle["bbox"],
            )
            helmet_status, helmet_reason, helmet_detections = judge_helmet(
                image_rgb, person["bbox"], video_name=video_name, frame_id=frame_id, pair_idx=pair_idx
            )

            anomaly_types = []
            if sidewalk_status in {"sidewalk_riding", "sidewalk_candidate"}:
                anomaly_types.append(sidewalk_status)
            if helmet_status == "no_helmet":
                anomaly_types.append("no_helmet")

            if sidewalk_status == "sidewalk_riding":
                loc_text = "인도 주행 의심"
            elif sidewalk_status == "sidewalk_candidate":
                loc_text = "인도 주행 후보"
            elif sidewalk_status == "road_riding":
                loc_text = "차도 주행"
            else:
                loc_text = "주행 위치 불확실"

            if helmet_status == "no_helmet":
                helmet_text = "헬멧 미착용 의심"
            elif helmet_status == "helmet":
                helmet_text = "헬멧 착용"
            else:
                helmet_text = "헬멧 판정 불확실"

            results.append({
                "video_name": video_name,
                "json_file": json_path.name,
                "frame_id": frame_id,
                "pair_index": pair_idx,
                "person": person,
                "vehicle": vehicle,
                "match": {
                    "score": pair["match_score"],
                    "overlap": pair["overlap"],
                    "distance": pair["distance"],
                },
                "sidewalk_status": sidewalk_status,
                "sidewalk_reason": sidewalk_reason,
                "road_score": round(road_score, 4),
                "sidewalk_score": round(sidewalk_score, 4),
                "helmet_status": helmet_status,
                "helmet_reason": helmet_reason,
                "helmet_detections": helmet_detections,
                "anomaly_types": anomaly_types,
                "description": f"{loc_text}, {helmet_text}",
            })

        if processed_pair_frames % 5 == 0:
            print(
                f"  pair_frames={processed_pair_frames}, frame_id={frame_id}, "
                f"pairs={video_pair_count}, total_results={len(results)}, "
                f"elapsed={round(time.time() - t0, 1)}s"
            )

        del image_rgb, pred

    cap.release()

    print(
        f"done {video_name}: pair_frames={processed_pair_frames}, "
        f"pairs={video_pair_count}, total_results={len(results)}, "
        f"elapsed={round(time.time() - t0, 1)}s"
    )

    if json_idx % SAVE_PARTIAL_EVERY_N_VIDEOS == 0:
        partial_path = OUT_DIR / "frame_anomaly_results_partial.json"
        with open(partial_path, "w", encoding="utf-8") as f:
            json.dump({"results": results, "failed_videos": failed_videos}, f, ensure_ascii=False, indent=2)
        print("partial saved:", partial_path)

    if DEVICE.type == "cuda" and json_idx % CLEAR_CUDA_CACHE_EVERY_N_VIDEOS == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("\nprocessed results:", len(results))
print("failed videos:", len(failed_videos))
print("sidewalk status:", Counter(r["sidewalk_status"] for r in results))
print("helmet status:", Counter(r["helmet_status"] for r in results))
print("total elapsed sec:", round(time.time() - start_time_all, 1))



[1/10] start bs_e4387_c1.mp4, frames=181, device=cuda, helmet_backend=none


C:\Users\우성한\AppData\Local\Temp\ipykernel_25832\2210232823.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  pair_frames=5, frame_id=235, pairs=5, total_results=5, elapsed=4.0s
  pair_frames=10, frame_id=260, pairs=10, total_results=10, elapsed=5.8s
  pair_frames=15, frame_id=295, pairs=15, total_results=15, elapsed=7.4s
done bs_e4387_c1.mp4: pair_frames=17, pairs=17, total_results=17, elapsed=8.0s

[2/10] start bs_e4387_c2.mp4, frames=180, device=cuda, helmet_backend=none
done bs_e4387_c2.mp4: pair_frames=2, pairs=2, total_results=19, elapsed=0.7s

[3/10] start es_e3768_c1.mp4, frames=181, device=cuda, helmet_backend=none
  pair_frames=5, frame_id=205, pairs=5, total_results=24, elapsed=1.7s
  pair_frames=10, frame_id=565, pairs=12, total_results=31, elapsed=3.3s
done es_e3768_c1.mp4: pair_frames=14, pairs=16, total_results=35, elapsed=4.6s

[4/10] start es_e3768_c2.mp4, frames=183, device=cuda, helmet_backend=none
  pair_frames=5, frame_id=175, pairs=5, total_results=40, elapsed=1.8s
  pair_frames=10, frame_id=215, pairs=11, total_results=46, elapsed=3.6s
  pair_frames=15, frame_id=275, p

In [14]:
# Cell 12. 구간 병합: C파트 검색용 segment 생성

def merge_segments_light(items, status_key, target_statuses, max_gap=10):
    if isinstance(target_statuses, str):
        target_statuses = {target_statuses}
    else:
        target_statuses = set(target_statuses)

    grouped = defaultdict(list)
    for item in items:
        if item.get(status_key) in target_statuses:
            grouped[item["video_name"]].append(int(item["frame_id"]))

    segments = []
    for video_name, fids in grouped.items():
        fids = sorted(set(fids))
        if not fids:
            continue
        start = fids[0]
        end = fids[0]
        count = 1
        for fid in fids[1:]:
            if fid - end <= max_gap:
                end = fid
                count += 1
            else:
                segments.append({"video_name": video_name, "start_frame": start, "end_frame": end, "count": count})
                start = fid
                end = fid
                count = 1
        segments.append({"video_name": video_name, "start_frame": start, "end_frame": end, "count": count})
    return segments

sidewalk_segments = merge_segments_light(results, "sidewalk_status", {"sidewalk_riding", "sidewalk_candidate"})
no_helmet_segments = merge_segments_light(results, "helmet_status", "no_helmet")

segments = []
for s in sidewalk_segments:
    s = dict(s)
    s["event_type"] = "sidewalk_riding"
    s["description"] = "인도 주행 또는 인도 주행 후보 구간"
    segments.append(s)
for s in no_helmet_segments:
    s = dict(s)
    s["event_type"] = "no_helmet"
    s["description"] = "헬멧 미착용 의심 구간"
    segments.append(s)

print("sidewalk segments:", len(sidewalk_segments))
print("no_helmet segments:", len(no_helmet_segments))
print("total segments:", len(segments))


sidewalk segments: 16
no_helmet segments: 0
total segments: 16


In [15]:
# Cell 13. 결과 저장
output = {
    "meta": {
        "input_cf2_dir": str(CF2_DIR),
        "video_root": str(VIDEO_ROOT),
        "device": str(DEVICE),
        "seg_model_id": SEG_MODEL_ID,
        "helmet_backend": helmet_backend,
        "helmet_model_path": None if HELMET_MODEL_PATH is None else str(HELMET_MODEL_PATH),
        "roboflow_model_id": ROBOFLOW_MODEL_ID if rf_hosted_ready else None,
        "road_id": ROAD_ID,
        "sidewalk_id": SIDEWALK_ID,
        "mask_patch_radius": MASK_PATCH_RADIUS,
        "sidewalk_judge_version": SIDEWALK_JUDGE_VERSION,
        "sidewalk_gt_prefixes": SIDEWALK_GT_PREFIXES,
        "helmet_gt_prefixes": HELMET_GT_PREFIXES,
        "dilate_kernel_size": DILATE_KERNEL_SIZE,
        "sidewalk_score_threshold": SIDEWALK_SCORE_THRESHOLD,
        "road_score_threshold": ROAD_SCORE_THRESHOLD,
    },
    "summary": {
        "processed_results": len(results),
        "failed_video_count": len(failed_videos),
        "sidewalk_status": dict(Counter(r["sidewalk_status"] for r in results)),
        "helmet_status": dict(Counter(r["helmet_status"] for r in results)),
        "anomaly_types": dict(Counter(t for r in results for t in r.get("anomaly_types", []))),
    },
    "failed_videos": failed_videos,
    "results": results,
}

frame_out = OUT_DIR / "frame_anomaly_results.json"
segment_out = OUT_DIR / "segment_anomaly_results.json"

with open(frame_out, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

with open(segment_out, "w", encoding="utf-8") as f:
    json.dump({"meta": output["meta"], "segments": segments}, f, ensure_ascii=False, indent=2)

print("saved:", frame_out)
print("saved:", segment_out)
print("summary:", json.dumps(output["summary"], ensure_ascii=False, indent=2))


saved: C:\Users\우성한\cf6_output\frame_anomaly_results.json
saved: C:\Users\우성한\cf6_output\segment_anomaly_results.json
summary: {
  "processed_results": 554,
  "failed_video_count": 0,
  "sidewalk_status": {
    "road_riding": 314,
    "sidewalk_riding": 120,
    "unknown": 110,
    "sidewalk_candidate": 10
  },
  "helmet_status": {
    "unknown": 554
  },
  "anomaly_types": {
    "sidewalk_riding": 120,
    "sidewalk_candidate": 10
  }
}


## cf5 대비 cf6 평가 셀

아래 셀은 cf6 결과를 `cf5_output/frame_anomaly_results.json`과 비교해서 인도주행 감지가 증가했는지/감소했는지 확인한다.


In [16]:
# Cell 14B. cf5 대비 cf6 평가: 인도주행 감지 증가/감소 확인
import json
from collections import defaultdict, Counter
from pathlib import Path


def load_result_json(path):
    path = Path(path)
    if not path.exists():
        print("missing:", path)
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return {"results": data, "summary": {}}
    return data


def group_name_from_video(video_name: str):
    if video_name.startswith(SIDEWALK_GT_PREFIXES):
        return "sidewalk_gt_group"
    if video_name.startswith(HELMET_GT_PREFIXES):
        return "helmet_gt_group"
    return "other_group"


def build_group_stats(data):
    stats = defaultdict(lambda: {
        "total": 0,
        "sidewalk_status": Counter(),
        "helmet_status": Counter(),
        "anomaly_types": Counter(),
        "detected_sidewalk_anomaly": 0,
    })
    if not data:
        return stats

    for r in data.get("results", []):
        video = r.get("video_name", "unknown_video")
        group = group_name_from_video(video)
        st = stats[group]
        st["total"] += 1
        st["sidewalk_status"][r.get("sidewalk_status", "missing")] += 1
        st["helmet_status"][r.get("helmet_status", "missing")] += 1
        for a in r.get("anomaly_types", []):
            st["anomaly_types"][a] += 1
        if r.get("sidewalk_status") in {"sidewalk_riding", "sidewalk_candidate"}:
            st["detected_sidewalk_anomaly"] += 1
    return stats


def build_video_stats(data):
    stats = defaultdict(lambda: {
        "total": 0,
        "sidewalk_status": Counter(),
        "detected_sidewalk_anomaly": 0,
    })
    if not data:
        return stats

    for r in data.get("results", []):
        video = r.get("video_name", "unknown_video")
        st = stats[video]
        st["total"] += 1
        st["sidewalk_status"][r.get("sidewalk_status", "missing")] += 1
        if r.get("sidewalk_status") in {"sidewalk_riding", "sidewalk_candidate"}:
            st["detected_sidewalk_anomaly"] += 1
    return stats


def summarize_group_stats(stats, label):
    print("\n" + "=" * 50)
    print(label)
    for group in ["sidewalk_gt_group", "helmet_gt_group", "other_group"]:
        st = stats.get(group)
        if not st:
            continue
        total = st["total"]
        detected = st["detected_sidewalk_anomaly"]
        recall_like = detected / total if total else 0.0
        print("\n------------------------------")
        print("group:", group)
        print("total:", total)
        print("sidewalk_status:", dict(st["sidewalk_status"]))
        print("detected_sidewalk_anomaly:", detected)
        print("recall_like:", round(recall_like, 4))
        print("helmet_status:", dict(st["helmet_status"]))


def compare_group_stats(old_stats, new_stats):
    print("\n" + "=" * 50)
    print("cf5 -> cf6 변화량")
    for group in ["sidewalk_gt_group", "helmet_gt_group", "other_group"]:
        old = old_stats.get(group, {"total": 0, "detected_sidewalk_anomaly": 0, "sidewalk_status": Counter()})
        new = new_stats.get(group, {"total": 0, "detected_sidewalk_anomaly": 0, "sidewalk_status": Counter()})

        old_total = old["total"]
        new_total = new["total"]
        old_det = old["detected_sidewalk_anomaly"]
        new_det = new["detected_sidewalk_anomaly"]
        old_rate = old_det / old_total if old_total else 0.0
        new_rate = new_det / new_total if new_total else 0.0

        print("\n------------------------------")
        print("group:", group)
        print("detected:", old_det, "->", new_det, "delta:", new_det - old_det)
        print("recall_like:", round(old_rate, 4), "->", round(new_rate, 4), "delta:", round(new_rate - old_rate, 4))
        print("sidewalk_status delta:")
        keys = sorted(set(old["sidewalk_status"].keys()) | set(new["sidewalk_status"].keys()))
        for k in keys:
            print(f"  {k}: {old['sidewalk_status'].get(k, 0)} -> {new['sidewalk_status'].get(k, 0)} ({new['sidewalk_status'].get(k, 0) - old['sidewalk_status'].get(k, 0):+d})")


def compare_video_stats(old_video_stats, new_video_stats):
    print("\n" + "=" * 50)
    print("영상별 cf5 -> cf6 변화량")
    videos = sorted(set(old_video_stats.keys()) | set(new_video_stats.keys()))
    for video in videos:
        old = old_video_stats.get(video, {"total": 0, "detected_sidewalk_anomaly": 0, "sidewalk_status": Counter()})
        new = new_video_stats.get(video, {"total": 0, "detected_sidewalk_anomaly": 0, "sidewalk_status": Counter()})
        old_total = old["total"]
        new_total = new["total"]
        old_det = old["detected_sidewalk_anomaly"]
        new_det = new["detected_sidewalk_anomaly"]
        old_rate = old_det / old_total if old_total else 0.0
        new_rate = new_det / new_total if new_total else 0.0
        print(
            f"{video}: detected {old_det}->{new_det} ({new_det - old_det:+d}), "
            f"rate {old_rate:.4f}->{new_rate:.4f} ({new_rate - old_rate:+.4f}), "
            f"status {dict(new['sidewalk_status'])}"
        )


cf5_data = load_result_json(BASELINE_CF5_RESULT_PATH)
cf6_data = load_result_json(OUT_DIR / "frame_anomaly_results.json")

cf5_group_stats = build_group_stats(cf5_data)
cf6_group_stats = build_group_stats(cf6_data)

summarize_group_stats(cf5_group_stats, "baseline cf5")
summarize_group_stats(cf6_group_stats, "current cf6")
compare_group_stats(cf5_group_stats, cf6_group_stats)

cf5_video_stats = build_video_stats(cf5_data)
cf6_video_stats = build_video_stats(cf6_data)
compare_video_stats(cf5_video_stats, cf6_video_stats)

print("\n판단 기준:")
print("- sidewalk_gt_group(ms/bs/es): detected와 recall_like가 증가하면 개선")
print("- helmet_gt_group(md), other_group: detected가 크게 증가하면 인도 오탐 증가 가능성")
print("- cf6 목표: sidewalk_gt_group 증가 + md/other 오탐 최소 유지")


missing: C:\Users\우성한\cf5_output\frame_anomaly_results.json

baseline cf5

current cf6

------------------------------
group: sidewalk_gt_group
total: 220
sidewalk_status: {'road_riding': 85, 'sidewalk_riding': 120, 'unknown': 6, 'sidewalk_candidate': 9}
detected_sidewalk_anomaly: 129
recall_like: 0.5864
helmet_status: {'unknown': 220}

------------------------------
group: helmet_gt_group
total: 204
sidewalk_status: {'road_riding': 105, 'unknown': 99}
detected_sidewalk_anomaly: 0
recall_like: 0.0
helmet_status: {'unknown': 204}

------------------------------
group: other_group
total: 130
sidewalk_status: {'road_riding': 124, 'unknown': 5, 'sidewalk_candidate': 1}
detected_sidewalk_anomaly: 1
recall_like: 0.0077
helmet_status: {'unknown': 130}

cf5 -> cf6 변화량

------------------------------
group: sidewalk_gt_group
detected: 0 -> 129 delta: 129
recall_like: 0.0 -> 0.5864 delta: 0.5864
sidewalk_status delta:
  road_riding: 0 -> 85 (+85)
  sidewalk_candidate: 0 -> 9 (+9)
  sidewalk_ridi

In [17]:
# Cell 14. 자연어 질의 테스트: 규칙 기반 C파트 프로토타입

def search_results(query: str, limit=10):
    wants_sidewalk = any(k in query for k in ["인도", "보도", "sidewalk"])
    wants_no_helmet = any(k in query for k in ["헬멧 안", "헬멧 미착용", "노헬멧", "no helmet", "no_helmet"])
    wants_road = any(k in query for k in ["차도", "road"])

    hits = []
    for r in results:
        ok = False
        if wants_sidewalk and r.get("sidewalk_status") in {"sidewalk_riding", "sidewalk_candidate"}:
            ok = True
        if wants_no_helmet and r.get("helmet_status") == "no_helmet":
            ok = True
        if wants_road and r.get("sidewalk_status") == "road_riding":
            ok = True
        if ok:
            hits.append(r)

    print("QUERY:", query)
    print("hits:", len(hits))
    for h in hits[:limit]:
        print({
            "video_name": h["video_name"],
            "frame_id": h["frame_id"],
            "sidewalk_status": h["sidewalk_status"],
            "sidewalk_score": h.get("sidewalk_score"),
            "road_score": h.get("road_score"),
            "helmet_status": h["helmet_status"],
            "description": h["description"],
        })
    return hits

# 예시
# search_results("인도에서 주행한 자전거 찾아줘")
# search_results("헬멧 미착용 오토바이 찾아줘")


In [18]:
# Cell 15. SegFormer 마스크 품질 디버그: 원본/road/sidewalk/vehicle band 확인
import matplotlib.pyplot as plt

DEBUG_VIDEO_NAME = sample.get("video_name", "")
DEBUG_FRAME_ID = None

# object가 있는 첫 frame을 자동 선택
for fr in sample.get("frames", []):
    persons, vehicles = extract_objects(fr)
    if persons and vehicles:
        DEBUG_FRAME_ID = int(fr.get("frame_id"))
        break

if DEBUG_VIDEO_NAME and DEBUG_FRAME_ID is not None:
    video_path = find_video_path(DEBUG_VIDEO_NAME)
    cap = cv2.VideoCapture(str(video_path))
    image = read_frame_from_cap(cap, DEBUG_FRAME_ID)
    cap.release()

    if image is None:
        raise RuntimeError(f"frame read failed: {DEBUG_VIDEO_NAME}, frame={DEBUG_FRAME_ID}")

    pred = segment_frame(image)
    road, sidewalk, road_d, sidewalk_d = make_class_masks(pred)

    stem = Path(DEBUG_VIDEO_NAME).stem
    original_path = DEBUG_DIR / f"{stem}_frame_{DEBUG_FRAME_ID}_original.jpg"
    road_path = DEBUG_DIR / f"{stem}_frame_{DEBUG_FRAME_ID}_road_mask.jpg"
    sidewalk_path = DEBUG_DIR / f"{stem}_frame_{DEBUG_FRAME_ID}_sidewalk_mask.jpg"
    sidewalk_d_path = DEBUG_DIR / f"{stem}_frame_{DEBUG_FRAME_ID}_sidewalk_dilated_mask.jpg"

    cv2.imwrite(str(original_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(road_path), (road.astype(np.uint8) * 255))
    cv2.imwrite(str(sidewalk_path), (sidewalk.astype(np.uint8) * 255))
    cv2.imwrite(str(sidewalk_d_path), (sidewalk_d.astype(np.uint8) * 255))

    print("debug video:", DEBUG_VIDEO_NAME)
    print("debug frame:", DEBUG_FRAME_ID)
    print("saved:", original_path)
    print("saved:", road_path)
    print("saved:", sidewalk_path)
    print("saved:", sidewalk_d_path)
else:
    print("디버그할 rider/vehicle frame을 찾지 못했습니다.")


C:\Users\우성한\AppData\Local\Temp\ipykernel_25832\2210232823.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


debug video: bs_e4387_c1.mp4
debug frame: 75
saved: C:\Users\우성한\cf6_debug\bs_e4387_c1_frame_75_original.jpg
saved: C:\Users\우성한\cf6_debug\bs_e4387_c1_frame_75_road_mask.jpg
saved: C:\Users\우성한\cf6_debug\bs_e4387_c1_frame_75_sidewalk_mask.jpg
saved: C:\Users\우성한\cf6_debug\bs_e4387_c1_frame_75_sidewalk_dilated_mask.jpg


In [19]:
# Cell 16. CUDA 메모리 확인
if torch.cuda.is_available():
    print(torch.cuda.is_available())
    print(torch.cuda.get_device_name(0))
    print("allocated:", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
    print("reserved:", round(torch.cuda.memory_reserved()/1024**3, 2), "GB")
else:
    print("CUDA not available")


True
NVIDIA GeForce RTX 3060
allocated: 0.32 GB
reserved: 2.65 GB
